In [ ]:
from utilities import initialize_with_module

await initialize_with_module('AITW-microstructures/1.2.0-foss-2023a', '2023.06')

In [ ]:
import utilities
from common import WOOD_MS_DEFAULTS
from wood_microstructure.cli.generate import generate

widget_generate, generate_event = utilities.click_cli_widget(
    generate,
    override_defaults=WOOD_MS_DEFAULTS,
    hide_params=['fit_porosity', 'fit_porosity_config'],
    title='Generate wood microstructure',
    button_description='Generate',
    button_running_text='Generating...'
)

# display(widget_generate)

In [ ]:
# Rewrite of the above cell with autodetection of which folders contains the images
import re
import os
import json
import ipywidgets as wdg
from IPython.display import display
from IPython.display import Image as IPyImage

from common import WOOD_MS_OUTDIR

rgx_dir = re.compile(r"Save(\w+)_(\d+)")
rgx_img = re.compile(r"volImgRef_(\d+).png")

# title = wdg.HTML(value="<h2>Microstructure Output Visualization</h2>")

def find_outputs(base_path=WOOD_MS_OUTDIR):
    outputs = []
    if not os.path.exists(base_path):
        return outputs
    for path in os.listdir(base_path):
        if not os.path.isdir(os.path.join(base_path, path)):
            continue
        match = rgx_dir.match(path)
        if not match:
            continue
        material, index = match.groups()
        outputs.append((material, int(index), os.path.join(base_path, path)))

    return outputs

# Start with a choice widget to select which output to visualize
output_selector = wdg.Dropdown(
    options=[],
    description='Select Output:',
    disabled=False,
    style={'description_width': '150px'},
)
output_selector.layout.width = '500px'

def update_output_options(*args):
    """Update the options of the output selector based on the available outputs in the results folder."""
    outputs = find_outputs()
    options = []
    for material, index, path in sorted(outputs):
        params = json.load(open(os.path.join(path, "params.json")))
        surrogate = params['surrogate']
        with_ray_cell = bool(params['is_exist_ray_cell'])
        with_vessel = bool(params['is_exist_vessel'])
        options.append((
            f"{material} - {index}: surrogate={surrogate}, ray_cells={with_ray_cell}, vessels={with_vessel}",
            path
        ))
    output_selector.options = options
    if output_selector.value is None:
        output_selector.value = options[0][1] if options else None

# Observe changes in the common event to update the output options whenever a new microstructure is generated
try:
    generate_event.observe(update_output_options, names='value')
except Exception as e:
    print("common-event not defined.")
    print("Re-launch this cell after executing the previous one to enable automatic updates of the dropdown choices.")

# Select whether to show the local distortion images or the backbone images, and a slider to select which slice to show
toggle_checkbox = wdg.Checkbox(
    value=False,
    description='With local distortion',
    style={'description_width': '150px'},
)
toggle_checkbox.layout.width = '500px'

# Slider to select which slice to show, with options automatically updated based on the selected output
image_slider = wdg.SelectionSlider(
    options=[0],
    description='Slice number:',
    continuous_update=False,
    style={'description_width': '150px'},
)
image_slider.layout.width = '500px'

image_output = wdg.Output()

def update_image_options(*args):
    """Update the options of the image slider based on the selected output and toggle state."""
    selected_output = output_selector.value
    if not selected_output:
        return
    imgdir = "LocalDistVolume" if toggle_checkbox.value else "volImgBackBone"
    custom_index = []
    for file in os.listdir(os.path.join(selected_output, imgdir)):
        match = rgx_img.match(file)
        if match:
            custom_index.append(int(match.group(1)))
    custom_index.sort()

    image_slider.options = custom_index
    image_slider.value = image_slider.options[0] if image_slider.options else None
output_selector.observe(update_image_options, names='value')

def display_image(*args):
    """Display the selected image based on the selected output, toggle state, and slider value."""
    selected_output = output_selector.value
    if not selected_output:
        return

    imgdir = "LocalDistVolume" if toggle_checkbox.value else "volImgBackBone"
    index_list = image_slider.options
    image_list = [os.path.join(selected_output, imgdir, f"volImgRef_{idx:05d}.png") for idx in index_list]

    selected_custom_index = image_slider.value
    position = index_list.index(selected_custom_index)

    with image_output:
        image_output.clear_output(wait=True)
        display(IPyImage(url=image_list[position]))

# Observe changes in the toggle and slider to update the displayed image
toggle_checkbox.observe(display_image, names='value')
image_slider.observe(display_image, names='value')
output_selector.observe(display_image, names='value')

# Initial update of the output options and image options
update_output_options()

visualizer_box = wdg.VBox([output_selector, toggle_checkbox, image_slider, image_output])
visualizer_accordion = wdg.Accordion(children=[visualizer_box], selected_index=None)
visualizer_accordion.set_title(0, "Microstructure Output Visualization")
main_box = wdg.VBox([widget_generate, visualizer_accordion])
main_accordion = wdg.Accordion(children=[main_box], selected_index=0)
main_accordion.set_title(0, "Generate Microstructure")
# main_vbox = wdg.VBox([output_selector, toggle_checkbox, image_slider, image_output])
# main_accordion = wdg.Accordion(children=[main_vbox], selected_index=None)
# main_accordion.set_title(0, "Microstructure Output Visualization")

display(main_accordion)

In [ ]:
import ipywidgets as wdg
import utilities
from common import WOOD_MS_DEFAULTS_FILTER
from wood_microstructure.cli.postproc import filter_porosity

widget_filter, _ = utilities.click_cli_widget(
    filter_porosity,
    override_defaults=WOOD_MS_DEFAULTS_FILTER,
    visible_panel_names=['Options'],
    title='Filter porosity',
    button_description='Filter',
    button_running_text='Filtering...'
)
filter_accordion = wdg.Accordion(children=[widget_filter], selected_index=None)
filter_accordion.set_title(0, "Filter Porosity")
display(filter_accordion)

In [ ]:
import ipywidgets as wdg
import utilities
from common import WOOD_MS_DEFAULTS_TRAIN
from wood_microstructure.cli.train import train

widget_train, _ = utilities.click_cli_widget(
    train,
    title='Train surrogate model',
    override_defaults=WOOD_MS_DEFAULTS_TRAIN,
    visible_panel_names=['Options'],
    custom_groupings={
        None: ['train_dir']
    },
    button_description='Train',
    button_running_text='Training...',
)

train_accordion = wdg.Accordion(children=[widget_train], selected_index=None)
train_accordion.set_title(0, "Train Surrogate Model")
display(train_accordion)

In [ ]:
import ipywidgets as wdg
spacer = wdg.Box(
    layout=wdg.Layout(height="95vh", flex="0 0 auto")
)
display(spacer)